In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv('../data/processed/train.csv')
train.head()
# train.info()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,...,CabinX_C,CabinX_D,CabinX_E,CabinX_F,CabinX_G,CabinX_NA,CabinX_T,CabinZ_P,CabinZ_S,CabinZ_None
0,0.705882,0.000000,0.000000,0.000000,0.000000,0.000000,False,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-0.176471,1.257597,0.557914,1.039101,1.581836,1.025068,True,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1.823529,1.012446,1.982557,0.000000,2.209146,1.053439,False,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.352941,0.000000,1.734311,1.887707,2.033282,1.418542,False,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-0.647059,1.529570,1.032843,1.602261,1.589025,0.295837,True,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
X,y = train.drop('Transported',axis=1), train['Transported'].astype(bool)

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, train_y, test_y = train_test_split(X,y,random_state=42,train_size=0.7)

In [5]:
models = {}

In [6]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(X_train,train_y)
models['rf'] = rf

In [7]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=5000)
lr.fit(X_train,train_y)
models['lr'] = lr

In [8]:
from xgboost import XGBClassifier
xg = XGBClassifier()
xg.fit(X_train,train_y)
models['xg'] = xg

In [9]:
from sklearn.ensemble import HistGradientBoostingClassifier

hg = HistGradientBoostingClassifier()
hg.fit(X_train,train_y)
models['hgb'] = hg

In [10]:
from sklearn.model_selection import cross_val_score

In [11]:
for key in models.keys():
    res = cross_val_score(models[key],X,y,scoring='accuracy')
    print("Mean cv:",key,np.mean(res))
    print("STD cv:",key,np.std(res))

Mean cv: rf 0.7868415044822263
STD cv: rf 0.008612859412381942
Mean cv: lr 0.7740712457922261
STD cv: lr 0.008704656095076735
Mean cv: xg 0.7977700370105433
STD cv: xg 0.0109937776639117
Mean cv: hgb 0.7991512654588334
STD cv: hgb 0.013798280193429624


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

xg_params = {
    'n_estimators':     [200, 500, 700],
    'learning_rate':    [0.01, 0.05, 0.15, 0.45, 0.85],
    'max_depth':        [4, 5, 6, 8],
    'subsample':        [0.5, 0.7, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'reg_alpha':        [0.01, 0.15, 0.35, 0.85],
    'reg_lambda':       [0.5, 1.0, 2.5, 5.0],
    'min_child_weight': [1, 3, 5, 10],
}

xg_cv = RandomizedSearchCV(
    XGBClassifier(
        n_jobs=1,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42
    ),
    param_distributions=xg_params,
    n_iter=500,              # number of random combos to try
    scoring='roc_auc',
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

xg_cv.fit(X_train, train_y)

print(f'Best XGB CV: {xg_cv.best_score_:.4f}')
print('Params:', xg_cv.best_params_)
res = cross_val_score(xg_cv.best_estimator_,X,y,scoring='roc_auc')
print("mean cv_score:",np.mean(res))
print("std cv_score:",np.std(res))

Fitting 5 folds for each of 500 candidates, totalling 2500 fits


In [ ]:
from sklearn.model_selection import GridSearchCV

hgb_params = {
    'learning_rate':      [0.01, 0.05, 0.15, 0.45, 0.85],
    'max_iter':           [200, 300, 500, 700],
    'max_depth':          [4, 5, 6, 8, None],
    'min_samples_leaf':   [10, 20, 30, 50],
    'l2_regularization':  [0.0, 0.01, 0.1, 1.0],
    'max_bins':           [128, 255],
}

hgb_cv = RandomizedSearchCV(
    HistGradientBoostingClassifier(),
    param_distributions=hgb_params,
    n_iter=500,              # number of random combos to try
    scoring='roc_auc',
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

hgb_cv.fit(X_train,train_y)
print(f'Best HGB CV: {hgb_cv.best_score_:.4f}')
print('Params:', hgb_cv.best_params_)
res = cross_val_score(hgb_cv.best_estimator_,X,y,scoring='roc_auc')
print("mean cv_score:",np.mean(res))
print("std cv_score:",np.std(res))

Fitting 5 folds for each of 500 candidates, totalling 2500 fits
Best HGB CV: 0.8932
Params: {'min_samples_leaf': 50, 'max_iter': 700, 'max_depth': 6, 'max_bins': 128, 'learning_rate': 0.01, 'l2_regularization': 0.1}
mean cv_score: 0.8879247288127777
std cv_score: 0.007754640586239715


In [ ]:
test = pd.read_csv("../data/processed/test.csv")
passengerId = test["PassengerId"]
test = test.drop("PassengerId", axis=1)

In [ ]:
hgb = hgb_cv.best_estimator_

hgb.fit(X,y)

y_pred = hgb.predict(test)

sub_hgb = pd.DataFrame({
    'PassengerId': passengerId,
    'Survived': y_pred
})

sub_hgb.to_csv('../hgb_submission.csv', index=False)

NameError: name 'hgb_cv' is not defined

In [ ]:
xg = xg_cv.best_estimator_

xg.fit(X,y)

y_pred = xg.predict(test)

sub_xg = pd.DataFrame({
    'PassengerId': passengerId,
    'Survived': y_pred
})

sub_xg.to_csv('../xg_submission.csv', index=False)